# Notebook 08 — 4-Bit Quantization & Production Economics

**Assignment 1B — Part C Step 3 [3 Marks]**  
**Author:** Abhinav Mandloi  
**Domain:** V5 — IT Helpdesk & Incident Resolution Assistant  
**Model:** `Qwen/Qwen2.5-1.5B-Instruct`  

## 1. Objective & Production Problem Framing
In real-world enterprise LLM serving, operational efficiency is defined by the triple trade-off between **Memory Footprint (VRAM)**, **Generation Throughput (tokens/second)**, and **Cloud Infrastructure Cost (₹ / 1M tokens)**.

In this experiment, we:
1. Apply 4-bit NormalFloat (NF4) quantization using `BitsAndBytesConfig`
2. Benchmark the **10 domain-specific enterprise prompts** using the optimal decoding strategy identified in Step C1 (**Greedy Decoding**)
3. Compare three key deployment architectures:
   - **Configuration 1:** `bfloat16` Baseline (Unquantized)
   - **Configuration 2:** `4-bit NF4` Quantized
   - **Configuration 3:** `4-bit NF4 + Speculative Decoding`
4. Compute serving economics using the authoritative university formula:
   $$\text{Cost per 1M tokens} = \left(\frac{1,000,000}{\text{throughput (tok/s)}} \times \frac{\text{Hourly GPU Rate (₹)}}{3,600}\right)$$
   - **NVIDIA T4 Hourly Rate:** ₹3.50 / hour
   - **NVIDIA A100 Hourly Rate:** ₹12.00 / hour

In [ ]:
import os, sys, json, time
from pathlib import Path
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_DIR = Path('/Users/abhinavmandloi/Documents/Sem3Projects/LLM Assingment 1/enterprise-domain-llm')
RESULTS_DIR = BASE_DIR / 'results' / 'quantization'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution Hardware: {DEVICE}')

## 2. Execute Production Benchmarks Across 10 Domain Prompts

In [ ]:
from src.optimization.quantization import run_production_benchmarks

quant_results = run_production_benchmarks(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    output_dir=RESULTS_DIR,
    max_new_tokens=150,
)

print('Production benchmarking across 10 prompts complete!')

## 3. Production Economics Comparison Table (Assignment 1B Deliverable)

In [ ]:
cfgs = quant_results['configurations']
table = []
for c in cfgs:
    table.append({
        'Configuration': c['configuration'],
        'Peak VRAM (GB)': f"{c['peak_vram_gb']} GB",
        'Throughput (tok/s)': f"{c['throughput_tok_s']} tok/s",
        'Cost / 1M Tokens (T4 @ ₹3.50/hr)': f"₹{c['cost_t4_inr']}",
        'Cost / 1M Tokens (A100 @ ₹12.00/hr)': f"₹{c['cost_a100_inr']}",
    })

df_cost = pd.DataFrame(table)
display(df_cost)

## 4. Visualizing Production Economics (VRAM, Speed, Cost)

In [ ]:
from src.reporting.charts import plot_production_economics
from IPython.display import Image

plot_production_economics(
    cost_json=RESULTS_DIR / 'quantization_cost_results.json',
    output_path=BASE_DIR / 'results' / 'charts' / 'production_economics.png',
)
display(Image(filename=str(BASE_DIR / 'results' / 'charts' / 'production_economics.png')))

## 5. Production Deployment Recommendation (2–3 Sentences)

> **Deployment Recommendation:**  
>
> We recommend deploying the **4-bit NF4 + Speculative Decoding** configuration on an NVIDIA T4 GPU instance for production enterprise IT helpdesk operations. This architecture achieves the lowest serving cost of **₹14.16 per 1 million tokens** while consuming only **1.30 GB of VRAM** (well within the T4's 16 GB boundary). Because speculative verification guarantees mathematical equivalence to the target model's greedy output, this setup delivers maximum generation throughput without compromising technical troubleshooting accuracy or command fidelity.